# 🏆 All-In-One Unified Master Pipeline: Simple & Robust ML

This master notebook brings together the three strongest alternative ML paradigms in one single file:
1. **Pipeline 1: 10-Fold Regularized Linear Models** (Logistic L1/L2, SGD ElasticNet, Ridge)
2. **Pipeline 2: 10-Fold Balanced Sub-Sampling Bagging** (200 balanced sub-models, no SMOTE)
3. **Pipeline 3: 10-Fold Stacking Meta-Learner** (Linear + Shallow Tree combinations)
4. **Consensus Ensemble: Multi-Paradigm Rank Blending** (Combines all 3 paradigms via percentile rank-averaging!)

**Hardware:** 100% CPU Friendly (Full run finishes in ~3-5 minutes on standard CPU).


In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, skew, kurtosis
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, roc_auc_score

SEED = 42
N_FOLDS = 10
print('=' * 75)
print('  ALL-IN-ONE MASTER SIMPLE PIPELINE (PSTU DATATHON)')
print('=' * 75)


## 1. Load Data & Robust Preprocessing

In [ ]:
DATA_DIR = next(d for d in ['/kaggle/input/competitions/pstu-data-thon-2026-vol-1', 'pstu-data-thon-2026-vol-1', '.'] if os.path.exists(os.path.join(d, 'train.csv')))
train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

TARGET_COL = 'TARGET'
y = train_raw[TARGET_COL].copy()
test_ids = test_raw['id'].copy() if 'id' in test_raw.columns else pd.Series(range(len(test_raw)), name='id')
X_tr_raw = train_raw.drop(columns=[TARGET_COL])
X_te_raw = test_raw.drop(columns=['id']) if 'id' in test_raw.columns else test_raw.copy()

feat_cols = [c for c in X_tr_raw.columns if c.startswith('feat_')]
cat_cols  = X_tr_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols  = [c for c in feat_cols if c not in cat_cols]

X_num_tr = X_tr_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)
X_num_te = X_te_raw[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

# 1% - 99% Outlier Clipping
p_low = np.percentile(X_num_tr, 1, axis=0)
p_high = np.percentile(X_num_tr, 99, axis=0)
X_num_tr = pd.DataFrame(np.clip(X_num_tr.values, p_low, p_high), columns=num_cols)
X_num_te = pd.DataFrame(np.clip(X_num_te.values, p_low, p_high), columns=num_cols)

# Simple Row Statistics
row_tr = pd.DataFrame({'row_mean': X_num_tr.mean(axis=1), 'row_std': X_num_tr.std(axis=1), 'row_zero': (X_num_tr==0).sum(axis=1)})
row_te = pd.DataFrame({'row_mean': X_num_te.mean(axis=1), 'row_std': X_num_te.std(axis=1), 'row_zero': (X_num_te==0).sum(axis=1)})

# Categorical Frequency
df_cat_tr = pd.DataFrame(index=X_tr_raw.index)
df_cat_te = pd.DataFrame(index=X_te_raw.index)
for col in cat_cols:
    freq_map = pd.concat([X_tr_raw[col], X_te_raw[col]]).value_counts(normalize=True).to_dict()
    df_cat_tr[f"{col}_freq"] = X_tr_raw[col].map(freq_map).fillna(0).astype(np.float32)
    df_cat_te[f"{col}_freq"] = X_te_raw[col].map(freq_map).fillna(0).astype(np.float32)

scaler = RobustScaler()
X_tr_scaled = np.nan_to_num(scaler.fit_transform(pd.concat([X_num_tr, df_cat_tr, row_tr], axis=1)), nan=0.0).astype(np.float32)
X_te_scaled = np.nan_to_num(scaler.transform(pd.concat([X_num_te, df_cat_te, row_te], axis=1)), nan=0.0).astype(np.float32)
print(f'Preprocessed Feature Matrix: Train {X_tr_scaled.shape} | Test {X_te_scaled.shape}')


## 2. Execute Pipeline 1: 10-Fold Regularized Linear Models

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_linear = np.zeros(len(y), dtype=np.float32)
test_linear = np.zeros(len(X_te_scaled), dtype=np.float32)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
    X_tr, y_tr = X_tr_scaled[tr_idx], y.iloc[tr_idx].values
    X_va, y_va = X_tr_scaled[va_idx], y.iloc[va_idx].values
    
    # Ensemble of L2 Logistic + ElasticNet SGD
    m1 = LogisticRegression(C=0.05, penalty='l2', class_weight='balanced', max_iter=500, random_state=SEED)
    m2 = SGDClassifier(loss='log_loss', penalty='elasticnet', alpha=1e-3, l1_ratio=0.15, class_weight='balanced', max_iter=500, random_state=SEED)
    m1.fit(X_tr, y_tr)
    m2.fit(X_tr, y_tr)
    
    va_prob = (m1.predict_proba(X_va)[:, 1] + m2.predict_proba(X_va)[:, 1]) / 2.0
    te_prob = (m1.predict_proba(X_te_scaled)[:, 1] + m2.predict_proba(X_te_scaled)[:, 1]) / 2.0
    
    oof_linear[va_idx] = va_prob
    test_linear += te_prob / N_FOLDS

auc_p1 = roc_auc_score(y, oof_linear)
print(f'Pipeline 1 Complete! OOF ROC-AUC: {auc_p1:.5f} [{time.time() - t0:.1f}s]')


## 3. Execute Pipeline 2: 10-Fold Balanced Sub-Sampling Bagging (No SMOTE)

In [ ]:
N_BAGGING_ROUNDS = 20
oof_bagging = np.zeros(len(y), dtype=np.float32)
test_bagging = np.zeros(len(X_te_scaled), dtype=np.float32)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
    X_tr_fold, y_tr_fold = X_tr_scaled[tr_idx], y.iloc[tr_idx].values
    X_va_fold, y_va_fold = X_tr_scaled[va_idx], y.iloc[va_idx].values
    
    pos_idx = np.where(y_tr_fold == 1)[0]
    neg_idx = np.where(y_tr_fold == 0)[0]
    n_pos = len(pos_idx)
    
    va_pred_sum = np.zeros(len(X_va_fold), dtype=np.float32)
    te_pred_sum = np.zeros(len(X_te_scaled), dtype=np.float32)
    
    for b in range(N_BAGGING_ROUNDS):
        rng = np.random.default_rng(SEED + fold * 100 + b)
        sampled_neg = rng.choice(neg_idx, size=n_pos, replace=False)
        sub_idx = np.concatenate([pos_idx, sampled_neg])
        
        clf = LogisticRegression(C=0.1, penalty='l2', max_iter=300, random_state=SEED+b)
        clf.fit(X_tr_fold[sub_idx], y_tr_fold[sub_idx])
        
        va_pred_sum += clf.predict_proba(X_va_fold)[:, 1] / N_BAGGING_ROUNDS
        te_pred_sum += clf.predict_proba(X_te_scaled)[:, 1] / (N_BAGGING_ROUNDS * N_FOLDS)
        
    oof_bagging[va_idx] = va_pred_sum
    test_bagging += te_pred_sum

auc_p2 = roc_auc_score(y, oof_bagging)
print(f'Pipeline 2 Complete! OOF ROC-AUC: {auc_p2:.5f} [{time.time() - t0:.1f}s]')


## 4. Execute Pipeline 3: 10-Fold Stacking Meta-Learner

In [ ]:
base_models = {
    'L2_Log': lambda: LogisticRegression(C=0.05, class_weight='balanced', max_iter=300, random_state=SEED),
    'HistGBDT': lambda: HistGradientBoostingClassifier(max_depth=4, l2_regularization=5.0, class_weight='balanced', max_iter=100, random_state=SEED),
    'ExtraTrees': lambda: ExtraTreesClassifier(n_estimators=60, max_depth=7, min_samples_leaf=30, class_weight='balanced', n_jobs=-1, random_state=SEED)
}

oof_meta = np.zeros((len(y), len(base_models)), dtype=np.float32)
test_meta = np.zeros((len(X_te_scaled), len(base_models)), dtype=np.float32)

t0 = time.time()
for m_idx, (m_name, m_fn) in enumerate(base_models.items()):
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_scaled, y)):
        clf = m_fn()
        clf.fit(X_tr_scaled[tr_idx], y.iloc[tr_idx].values)
        oof_meta[va_idx, m_idx] = clf.predict_proba(X_tr_scaled[va_idx])[:, 1]
        test_meta[:, m_idx] += clf.predict_proba(X_te_scaled)[:, 1] / N_FOLDS

# Level-1 Meta Learner
oof_stack = np.zeros(len(y), dtype=np.float32)
test_stack = np.zeros(len(X_te_scaled), dtype=np.float32)
for fold, (tr_idx, va_idx) in enumerate(skf.split(oof_meta, y)):
    meta = LogisticRegression(C=1.0, max_iter=200, random_state=SEED)
    meta.fit(oof_meta[tr_idx], y.iloc[tr_idx].values)
    oof_stack[va_idx] = meta.predict_proba(oof_meta[va_idx])[:, 1]
    test_stack += meta.predict_proba(test_meta)[:, 1] / N_FOLDS

auc_p3 = roc_auc_score(y, oof_stack)
print(f'Pipeline 3 Complete! OOF ROC-AUC: {auc_p3:.5f} [{time.time() - t0:.1f}s]')


## 5. Multi-Paradigm Rank-Consensus Blending & F1 Optimization

In [ ]:
# Normalize predictions into percentile ranks (0.0 to 1.0)
rank_p1 = rankdata(oof_linear) / len(oof_linear)
rank_p2 = rankdata(oof_bagging) / len(oof_bagging)
rank_p3 = rankdata(oof_stack) / len(oof_stack)

# Master Consensus Rank Average
oof_consensus = (rank_p1 + rank_p2 + rank_p3) / 3.0

test_rank_p1 = rankdata(test_linear) / len(test_linear)
test_rank_p2 = rankdata(test_bagging) / len(test_bagging)
test_rank_p3 = rankdata(test_stack) / len(test_stack)
test_consensus = (test_rank_p1 + test_rank_p2 + test_rank_p3) / 3.0

# Function to optimize F1 threshold
def get_best_f1(oof_arr, y_true):
    thresholds = np.arange(0.05, 0.95, 0.001)
    best_f, best_t = 0.0, 0.5
    for t in thresholds:
        b = (oof_arr >= t).astype(int)
        if b.sum() == 0: continue
        f = f1_score(y_true, b)
        if f > best_f: best_f, best_t = f, t
    return best_f, best_t

f1_1, t_1 = get_best_f1(oof_linear, y)
f1_2, t_2 = get_best_f1(oof_bagging, y)
f1_3, t_3 = get_best_f1(oof_stack, y)
f1_c, t_c = get_best_f1(oof_consensus, y)

print('=' * 75)
print('  COMPUTED F1 SCORE SUMMARY TABLE')
print('=' * 75)
print(f"  1. Regularized Linear Ensemble:    OOF AUC = {auc_p1:.5f} | Peak F1 = {f1_1:.5f} @ t={t_1:.4f}")
print(f"  2. Balanced Bagging (No SMOTE):    OOF AUC = {auc_p2:.5f} | Peak F1 = {f1_2:.5f} @ t={t_2:.4f}")
print(f"  3. Two-Stage Stacking Meta-Learner: OOF AUC = {auc_p3:.5f} | Peak F1 = {f1_3:.5f} @ t={t_3:.4f}")
print(f"  4. 🏆 MASTER CONSENSUS BLEND:       OOF AUC = {roc_auc_score(y, oof_consensus):.5f} | Peak F1 = {f1_c:.5f} @ t={t_c:.4f}")
print('=' * 75)


## 6. Export All Submissions & Master Consensus

In [ ]:
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
os.makedirs(OUT_DIR, exist_ok=True)

# Save Individual Submissions
pd.DataFrame({'id': test_ids.values, 'TARGET': (test_linear >= t_1).astype(int)}).to_csv(os.path.join(OUT_DIR, 'submission_linear.csv'), index=False)
pd.DataFrame({'id': test_ids.values, 'TARGET': (test_bagging >= t_2).astype(int)}).to_csv(os.path.join(OUT_DIR, 'submission_bagging.csv'), index=False)
pd.DataFrame({'id': test_ids.values, 'TARGET': (test_stack >= t_3).astype(int)}).to_csv(os.path.join(OUT_DIR, 'submission_stacking.csv'), index=False)

# Save Master Consensus Submission (Primary)
pd.DataFrame({'id': test_ids.values, 'TARGET': (test_consensus >= t_c).astype(int)}).to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False)
pd.DataFrame({'id': test_ids.values, 'TARGET': test_consensus}).to_csv(os.path.join(OUT_DIR, 'submission_prob.csv'), index=False)

print(f'Successfully generated submission.csv (Master Consensus) with {(test_consensus >= t_c).sum():,} positive predictions!')
